# Binary Classification in Medicine - Predicting Heart Disease

**Dataset:** UCI Heart Disease (Janosi, Steinbrunn, Pfisterer & Detrano, 1988) — 920 patients from **4 hospitals**:
Cleveland (USA), Budapest (Hungary), Zurich/Basel (Switzerland), Long Beach VA (USA).

**Task:** predict whether a patient has angiographically confirmed coronary artery disease (>50% narrowing of a major vessel) from 13 clinical variables.

Why this dataset is good for teaching:
- small (920 rows × 14 columns) → everything runs in seconds
- **real missing values**, with very different rates across columns → perfect for the imputation / column-removal rule
- **4 hospitals** → a natural *group* variable to discuss generalisation to a new hospital
- mixed numeric + categorical features → scaling and one-hot encoding

### Roadmap
1. Load the data
2. Exploratory data analysis (EDA)
3. Train / test split
4. Missing values: drop columns with > 20% missing, impute median / mode
5. Preprocessing: imputation + scaling + encoding
6. Feature selection
7. Baseline models
8. Hyperparameter tuning — `GridSearchCV` and `RandomizedSearchCV`
9. Evaluation on the test set — ROC curve, confusion matrix, threshold choice
10. Bonus: leave-one-hospital-out validation
11. Exercises

> **Golden rule of the whole notebook:** the test set is locked in a drawer. Every decision (which columns to drop, the median used for imputation, the scaling parameters, the selected features, the hyperparameters) is learned **from the training set only**.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     cross_val_predict, GridSearchCV, RandomizedSearchCV)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, RocCurveDisplay, ConfusionMatrixDisplay,
                             confusion_matrix, classification_report, accuracy_score)
from scipy.stats import randint

RANDOM_STATE = 42
CLASS_COLORS = ["#4C72B0", "#C44E52"]   # blue = no disease, red = disease
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

# scikit-learn >= 1.8 prints deprecation notices for `penalty=`; the results are not affected
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Inconsistent values")

## 1. Load the data

The four hospital files are downloaded directly from the UCI repository. Missing values are coded as `?`.

| Column | Meaning |
|---|---|
| `age` | age (years) |
| `sex` | 1 = male, 0 = female |
| `cp` | chest pain type: 1 typical angina, 2 atypical angina, 3 non-anginal pain, 4 asymptomatic |
| `trestbps` | resting blood pressure (mm Hg) |
| `chol` | serum cholesterol (mg/dl) |
| `fbs` | fasting blood sugar > 120 mg/dl (1 = yes) |
| `restecg` | resting ECG: 0 normal, 1 ST-T abnormality, 2 left ventricular hypertrophy |
| `thalach` | maximum heart rate achieved |
| `exang` | exercise-induced angina (1 = yes) |
| `oldpeak` | ST depression induced by exercise |
| `slope` | slope of peak exercise ST segment: 1 up, 2 flat, 3 down |
| `ca` | number of major vessels coloured by fluoroscopy (0–3) |
| `thal` | thallium scan: 3 normal, 6 fixed defect, 7 reversible defect |
| `num` | diagnosis: 0 = no disease, 1–4 = disease severity |

> **If the download fails**: download the four `processed.*.data` files once from
> <https://archive.ics.uci.edu/dataset/45/heart+disease>, put them in a `data/` folder and set `BASE_URL = "data/"`.

In [ ]:
BASE_URL = "data/"
SITES = {"cleveland": "Cleveland", "hungarian": "Hungary",
         "switzerland": "Switzerland", "va": "VA Long Beach"}
COLS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
        "exang", "oldpeak", "slope", "ca", "thal", "num"]

frames = []
for file_key, site_name in SITES.items():
    d = pd.read_csv(f"{BASE_URL}processed.{file_key}.data", header=None,
                    names=COLS, na_values="?")
    d["site"] = site_name
    frames.append(d)
df = pd.concat(frames, ignore_index=True)

# Binary target: any disease (num 1-4) vs no disease (num 0)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

print(df.shape)
df.head()

## 2. Exploratory Data Analysis

### 2.1 Feature types
We group the columns once here and use these names for the rest of the notebook.

In [ ]:
NUMERIC = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
BINARY = ["sex", "fbs", "exang"]
CATEGORICAL = ["cp", "restecg", "slope", "thal"]   # nominal -> one-hot
TARGET = "target"
GROUP = "site"

### 2.2 Structure and summary statistics

In [ ]:
df.info()

In [ ]:
df.describe().T.round(2)

**Look carefully at the minimum of `chol` and `trestbps`.** A cholesterol of 0 mg/dl or a blood pressure of 0 mm Hg is physiologically impossible: these are **missing values in disguise**.
Data exploration is also checking that the numbers make clinical sense.

In [ ]:
print("chol == 0     :", (df["chol"] == 0).sum(), "rows")
print("trestbps == 0 :", (df["trestbps"] == 0).sum(), "rows")
print("\nchol == 0 by site:")
print(df.loc[df["chol"] == 0, GROUP].value_counts())

# Recode impossible zeros as missing
df.loc[df["chol"] == 0, "chol"] = np.nan
df.loc[df["trestbps"] == 0, "trestbps"] = np.nan

### 2.3 Target distribution - overall and per hospital

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df[TARGET].value_counts().sort_index().plot.bar(ax=axes[0], color=CLASS_COLORS)
axes[0].set_xticklabels(["No disease (0)", "Disease (1)"], rotation=0)
axes[0].set_ylabel("Number of patients")
axes[0].set_title("Class balance")

prev = df.groupby(GROUP)[TARGET].agg(["size", "mean"])
prev.columns = ["n", "prevalence"]
prev["prevalence"].sort_values().plot.barh(ax=axes[1], color="#55A868")
axes[1].set_xlim(0, 1)
axes[1].set_title("Disease prevalence per hospital")

plt.tight_layout()
plt.show()
prev.round(2)

The classes are roughly balanced overall, but prevalence varies hugely between hospitals (Switzerland is almost all positive).
Hospitals differ in referral patterns, populations and measurement protocols - keep this in mind for sections 3 and 10.

### 2.4 Missing values

In [ ]:
miss = df.drop(columns=[TARGET, GROUP]).isna().mean().sort_values(ascending=False) * 100

ax = miss.plot.bar(figsize=(10, 4), color="#8172B2")
ax.axhline(20, color="red", ls="--", label="20% threshold")
ax.set_ylabel("% missing")
ax.set_title("Missing values per column")
ax.legend()
plt.show()

In [ ]:
miss_site = df.drop(columns=TARGET).groupby(GROUP).apply(lambda g: g.isna().mean() * 100)

plt.figure(figsize=(10, 3.5))
sns.heatmap(miss_site, annot=True, fmt=".0f", cmap="Reds", cbar_kws={"label": "% missing"})
plt.title("% missing per column and hospital")
plt.show()

`ca`, `thal` and `slope` are mostly missing, and the missingness is concentrated in specific hospitals (some tests were simply not performed there).
This is *missing not at random* - an important clinical point: imputing a test that was never done can hide real differences between sites.

### 2.5 Distributions of numeric features by class

In [ ]:
num_plot = ["age", "trestbps", "chol", "thalach", "oldpeak"]

fig, axes = plt.subplots(1, len(num_plot), figsize=(18, 3.5))
for ax, col in zip(axes, num_plot):
    sns.histplot(data=df, x=col, hue=TARGET, kde=True, ax=ax, element="step",
                 stat="density", common_norm=False, palette=CLASS_COLORS)
    ax.set_title(col)
plt.tight_layout()
plt.show()

### 2.6 Binary and categorical features: counts by class

In [ ]:
cat_plot = ["sex", "cp", "fbs", "restecg", "exang", "slope"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), cat_plot):
    counts = pd.crosstab(df[col], df[TARGET])
    counts.plot.bar(stacked=True, ax=ax, color=CLASS_COLORS, rot=0)
    ax.set_title(col)
    ax.set_ylabel("Count")
    ax.legend(["No disease", "Disease"])
plt.tight_layout()
plt.show()

### 2.7 Correlations

In [ ]:
corr = df.drop(columns=GROUP).corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", square=True)
plt.title("Pearson correlation")
plt.show()

## 3. Train / test split

### Why groups matter
Sometimes several rows belong to the same *group* and must stay on the same side of the split.
- If a patient had several visits (several rows), the group would be the patient ID - otherwise the model is tested on patients it has already seen (**data leakage**).
- Here each row is a different patient, so there is no patient-level leakage. 

So we use a random 80/20 split, stratified on **target × hospital** so both sets have the same mix.

In [ ]:
X = df.drop(columns=[TARGET, GROUP])
y = df[TARGET]

strata = y.astype(str) + "_" + df[GROUP]     # stratify on the combination of label and hospital
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=strata, random_state=RANDOM_STATE)

print(f"Train: {X_train.shape[0]} patients   Test: {X_test.shape[0]} patients")

## 4. Missing values

### 4.1 Drop columns with more than 20% missing - decided on the **training set only**

In [ ]:
MISSING_THRESHOLD = 0.20

def find_columns_to_drop(X):
    # Columns with more than MISSING_THRESHOLD missing values
    missing = X.isna().mean()
    return missing[missing > MISSING_THRESHOLD].index.tolist()

print("% missing in TRAINING set:")
print((X_train.isna().mean().sort_values(ascending=False) * 100).round(1).to_string())
print("\nColumns to drop:", find_columns_to_drop(X_train))

### 4.2 Imputation: median for numeric, mode for binary / categorical

Why the **median** and not the mean?
It is robust to outliers and skewed distributions (look at `oldpeak`).
Why the **mode** for binary / categorical?
A "mean" of chest-pain categories has no meaning.

In [ ]:
# Effect of median imputation: a spike at the median
col = "trestbps"
before = X_train[col]
after = X_train[col].fillna(X_train[col].median())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
axes[0].hist(before.dropna(), bins=30, color="#4C72B0")
axes[0].set_title(f"{col} - before")
axes[1].hist(after, bins=30, color="#DD8452")
axes[1].set_title(f"{col} - after median imputation")
plt.tight_layout()
plt.show()

Median imputation shrinks the variance and creates an artificial peak.
With a lot of missingness this distortion grows - another reason for the 20% rule.

## 5. Preprocessing: imputation + scaling + encoding

### Why scale?
Logistic regression with regularisation, SVMs and k-NN are sensitive to the **units** of the features (cholesterol in hundreds, `oldpeak` around 1).
`RobustScaler` subtracts the **median** and divides by the **interquartile range (IQR)**, so a few extreme values do not distort the scale. Tree-based models do not need scaling.

### Why one-hot encoding?
`cp = 4` is not "twice" `cp = 2`: categories have no order, so each category becomes its own 0/1 column.

### Fit on train, apply to both
Every step **learns** something (medians, modes, IQRs, list of categories). We learn it on the training set with `.fit_transform()`
and apply the *same* values to the test set with `.transform()`.

In [ ]:
# 1. Drop columns with too many missing values (decided on train)
cols_to_drop = find_columns_to_drop(X_train)
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

numeric = [c for c in NUMERIC if c in X_train.columns]
binary = [c for c in BINARY if c in X_train.columns]
categorical = [c for c in CATEGORICAL if c in X_train.columns]

# 2. Numeric columns: fill missing with the median, then scale
num_imputer = SimpleImputer(strategy="median")
scaler = RobustScaler()
num_train = scaler.fit_transform(num_imputer.fit_transform(X_train[numeric]))
num_test = scaler.transform(num_imputer.transform(X_test[numeric]))

# 3. Binary columns: fill missing with the most frequent value
bin_imputer = SimpleImputer(strategy="most_frequent")
bin_train = bin_imputer.fit_transform(X_train[binary])
bin_test = bin_imputer.transform(X_test[binary])

# 4. Categorical columns: fill missing with the most frequent value, then one-hot encode
cat_imputer = SimpleImputer(strategy="most_frequent")
encoder = OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False)
cat_train = encoder.fit_transform(cat_imputer.fit_transform(X_train[categorical]))
cat_test = encoder.transform(cat_imputer.transform(X_test[categorical]))

# 5. Put the three blocks side by side in one table
names = numeric + binary + list(encoder.get_feature_names_out(categorical))
Xt_train = pd.DataFrame(np.hstack([num_train, bin_train, cat_train]), columns=names, index=X_train.index)
Xt_test = pd.DataFrame(np.hstack([num_test, bin_test, cat_test]), columns=names, index=X_test.index)

feature_names = Xt_train.columns
n_features = len(feature_names)

print("Train:", Xt_train.shape, "  Test:", Xt_test.shape)
Xt_train.describe().T[["50%", "mean", "min", "max"]].round(2)

Numeric features now have median 0 (and comparable spread); binary and one-hot columns are 0/1. No missing values are left.

> **NB:** from now on, cross-validation runs on `Xt_train`, which was imputed and scaled using the *whole* training set.
> The validation folds therefore "saw" a little of the preprocessing, so CV scores are slightly optimistic.
> This does **not** affect the test set, which was never used to learn anything — so the final evaluation in section 9 stays honest.

## 6. Baseline models

Always compare with a trivial baseline: a classifier that ignores the features and predicts the majority class.
We use 5-fold stratified cross-validation on the training set; the same `cv` object is reused everywhere below.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baselines = {
    "Random model (dummy)": DummyClassifier(strategy="uniform"),
    "Logistic regression (default)": LogisticRegression(max_iter=1000),
    "Random forest (default)": RandomForestClassifier(random_state=RANDOM_STATE),
}
for name, clf in baselines.items():
    auc = cross_val_score(clf, Xt_train, y_train, cv=cv, scoring="roc_auc")
    acc = cross_val_score(clf, Xt_train, y_train, cv=cv, scoring="accuracy")
    print(f"{name:32s} ROC-AUC = {auc.mean():.3f} ± {auc.std():.3f}   accuracy = {acc.mean():.3f}")

## 7. Hyperparameter tuning

**Parameters** are learned from data (e.g. regression coefficients). **Hyperparameters** are set before training (e.g. regularisation strength `C`, number of trees, number of selected features `k`).
We choose them by **cross-validation on the training set**.

| | `GridSearchCV` | `RandomizedSearchCV` |
|---|---|---|
| How | tries **every** combination of a grid | samples `n_iter` random combinations from distributions |
| Cost | grows multiplicatively with each hyperparameter | fixed by `n_iter` |
| Best for | few hyperparameters, small grids | many hyperparameters, continuous ranges |

### 7.1 Grid search - logistic regression 
Run a grid search over `C` and the penalty.

In [ ]:
param_grid = {"C": [0.001, 0.01, 0.1, 1, 10, 100], "penalty": ["l1", "l2"]}

n_comb =  len(param_grid["C"]) * len(param_grid["penalty"])
print(f"{n_comb} combinations × {cv.get_n_splits()} folds = {n_comb * cv.get_n_splits()} fits\n")

all_results = []
best_score = 0

grid = GridSearchCV(LogisticRegression(solver="liblinear", max_iter=5000, random_state=RANDOM_STATE),
                    param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
grid.fit(Xt_train, y_train)
print(f"best CV ROC-AUC = {grid.best_score_:.3f}   {grid.best_params_}")

res = pd.DataFrame(grid.cv_results_)
all_results.append(res)

best_score = grid.best_score_
best_lr = grid.best_estimator_     # already refitted on the whole training set


### 7.2 Randomized search - random forest
A random forest does its own feature selection (each split picks the best feature), so here we give it all the features.

In [ ]:
param_dist = {
    "n_estimators": randint(100, 500),
    "max_depth": [None, 3, 5, 8, 12],
    "min_samples_leaf": randint(1, 20),
    "max_features": ["sqrt", "log2", None],
    "class_weight": [None, "balanced"],
}

rand = RandomizedSearchCV(RandomForestClassifier(random_state=RANDOM_STATE), param_dist,
                          n_iter=40, cv=cv, scoring="roc_auc", n_jobs=-1, random_state=RANDOM_STATE)
rand.fit(Xt_train, y_train)
best_rf = rand.best_estimator_

print("Best parameters:", rand.best_params_)
print(f"Best CV ROC-AUC: {rand.best_score_:.3f}")

In [ ]:
# How much does the random search explore? Score of each sampled configuration
r = pd.DataFrame(rand.cv_results_).sort_values("rank_test_score")

plt.figure(figsize=(8, 3.5))
plt.errorbar(range(len(r)), r["mean_test_score"], yerr=r["std_test_score"], fmt="o", ms=4, capsize=2)
plt.xlabel("configuration (sorted by rank)")
plt.ylabel("mean CV ROC-AUC")
plt.title("Randomized search for the 40 configurations")
plt.show()

r[["mean_test_score", "std_test_score", "param_max_depth",
   "param_min_samples_leaf", "param_max_features"]].head(5).round(3)

## 8. Final evaluation on the test set

Only now do we open the drawer. We evaluate each tuned model **once**.
Each model is stored with the test columns it needs (the logistic regression uses only the selected features).

In [ ]:
models = {
    "Logistic regression (grid)": (best_lr, Xt_test),
    "Random forest (random)": (best_rf, Xt_test),
}

rows = []
for name, (model, X_eval) in models.items():
    proba = model.predict_proba(X_eval)[:, 1]
    pred = (proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rows.append({"model": name,
                 "ROC-AUC": roc_auc_score(y_test, proba),
                 "accuracy": accuracy_score(y_test, pred),
                 "sensitivity (recall)": tp / (tp + fn),
                 "specificity": tn / (tn + fp),
                 "precision": tp / (tp + fp),
                 "NPV": tn / (tn + fn)})
pd.DataFrame(rows).set_index("model").round(3)

### 8.1 ROC curve

The ROC curve plots **sensitivity (true positive rate)** against **1 − specificity (false positive rate)** for every possible threshold.
The AUC is the probability that a randomly chosen diseased patient gets a higher score than a randomly chosen healthy one (0.5 = coin flip, 1 = perfect).

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
for name, (model, X_eval) in models.items():
    RocCurveDisplay.from_estimator(model, X_eval, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Chance (AUC = 0.5)")
ax.set_title("ROC curves - test set")
ax.legend(loc="lower right")
plt.show()

### 8.2 Confusion matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (name, (model, X_eval)) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_eval, y_test, display_labels=["No disease", "Disease"],
                                          cmap="Blues", ax=ax, colorbar=False)
    ax.grid(False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

print(classification_report(y_test, best_lr.predict(Xt_test),
                            target_names=["No disease", "Disease"]))

In medicine the two errors are not equal:
- **False negative** - a sick patient sent home (missed diagnosis)
- **False positive** - a healthy patient sent for an unnecessary invasive angiography

### 8.3 Choosing the threshold
0.5 is just a default. For a screening test we may prefer high sensitivity. **The threshold should be chosen on training data, not on the test set.**
Here we pick the threshold reaching ≥ 90% sensitivity using cross-validated predictions on the training set, then apply it to the test set.

In [ ]:
# 1. Choose the threshold on the TRAINING set (cross-validated predictions)
train_proba = cross_val_predict(best_lr, Xt_train, y_train, cv=cv, method="predict_proba")[:, 1]
fpr, tpr, thr = roc_curve(y_train, train_proba)

TARGET_SENS = 0.90
threshold = thr[np.argmax(tpr >= TARGET_SENS)]
print(f"Threshold for ≥{TARGET_SENS:.0%} sensitivity (from training CV): {threshold:.3f}")

# 2. Apply the default and the chosen threshold to the TEST set
test_proba = best_lr.predict_proba(Xt_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, t in zip(axes, [0.5, threshold]):
    cm = confusion_matrix(y_test, (test_proba >= t).astype(int))
    tn, fp, fn, tp = cm.ravel()
    ConfusionMatrixDisplay(cm, display_labels=["No disease", "Disease"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.grid(False)
    ax.set_title(f"threshold = {t:.2f}\nsensitivity = {tp/(tp+fn):.2f}, specificity = {tn/(tn+fp):.2f}")
plt.tight_layout()
plt.show()

In [ ]:
# Where does the chosen threshold sit on the test ROC curve?
fpr_test, tpr_test, _ = roc_curve(y_test, test_proba)
tn, fp, fn, tp = confusion_matrix(y_test, (test_proba >= threshold).astype(int)).ravel()

plt.figure(figsize=(6, 5.5))
plt.plot(fpr_test, tpr_test, label=f"Logistic regression (AUC = {roc_auc_score(y_test, test_proba):.2f})")
plt.scatter(fp / (fp + tn), tp / (tp + fn), s=80, c="red", zorder=3, label=f"chosen threshold {threshold:.2f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False positive rate (1 − specificity)")
plt.ylabel("True positive rate (sensitivity)")
plt.title("Operating point")
plt.legend(loc="lower right")
plt.show()

## 9. Bonus - Does the model work in a new hospital? (Leave-one-hospital-out)

So far, train and test came from the same 4 hospitals. Now we train on 3 hospitals and test on the 4th, rotating through all four.
In each round the whole procedure is repeated **on the 3 training hospitals only**: column dropping and preprocessing (our `preprocess` function), then tuning `C` with a grid search.

In [ ]:
def preprocess(X_train, X_test):
    # 1. Drop columns with too many missing values (decided on train)
    cols_to_drop = find_columns_to_drop(X_train)
    X_train = X_train.drop(columns=cols_to_drop)
    X_test = X_test.drop(columns=cols_to_drop)

    numeric = [c for c in NUMERIC if c in X_train.columns]
    binary = [c for c in BINARY if c in X_train.columns]
    categorical = [c for c in CATEGORICAL if c in X_train.columns]

    # 2. Numeric columns: fill missing with the median, then scale
    num_imputer = SimpleImputer(strategy="median")
    scaler = RobustScaler()
    num_train = scaler.fit_transform(num_imputer.fit_transform(X_train[numeric]))
    num_test = scaler.transform(num_imputer.transform(X_test[numeric]))

    # 3. Binary columns: fill missing with the most frequent value
    bin_imputer = SimpleImputer(strategy="most_frequent")
    bin_train = bin_imputer.fit_transform(X_train[binary])
    bin_test = bin_imputer.transform(X_test[binary])

    # 4. Categorical columns: fill missing with the most frequent value, then one-hot encode
    cat_imputer = SimpleImputer(strategy="most_frequent")
    encoder = OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False)
    cat_train = encoder.fit_transform(cat_imputer.fit_transform(X_train[categorical]))
    cat_test = encoder.transform(cat_imputer.transform(X_test[categorical]))

    # 5. Put the three blocks side by side in one table
    names = numeric + binary + list(encoder.get_feature_names_out(categorical))
    Xt_train = pd.DataFrame(np.hstack([num_train, bin_train, cat_train]), columns=names, index=X_train.index)
    Xt_test = pd.DataFrame(np.hstack([num_test, bin_test, cat_test]), columns=names, index=X_test.index)
    
    return Xt_train, Xt_test

In [ ]:
results = []
for hospital in df[GROUP].unique():
    is_test = df[GROUP] == hospital
    X_tr, X_te = X[~is_test], X[is_test]
    y_tr, y_te = y[~is_test], y[is_test]

    Xt_tr, Xt_te = preprocess(X_tr, X_te)
    grid = GridSearchCV(LogisticRegression(solver="liblinear", max_iter=5000, random_state=RANDOM_STATE),
                        param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
    grid.fit(Xt_tr, y_tr)

    proba = grid.predict_proba(Xt_te)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_te, (proba >= 0.5).astype(int), labels=[0, 1]).ravel()
    results.append({"test hospital": hospital,
                    "n": len(y_te),
                    "prevalence": y_te.mean(),
                    "ROC-AUC": roc_auc_score(y_te, proba),
                    "sensitivity": tp / (tp + fn),
                    "specificity": tn / (tn + fp),
                    "columns dropped": find_columns_to_drop(X_tr)})

pd.DataFrame(results).set_index("test hospital").round(2)


- Compare these AUCs with the random-split test AUC. Performance usually **drops** on an unseen hospital (*dataset shift*).
- Look at Switzerland: with ~94% prevalence and few negatives, specificity is estimated on a handful of patients - very unstable.
- The columns dropped change depending on which hospitals are in the training set: the 20% rule is itself learned from the data.
- This is why clinical prediction models need **external validation** before use.

## 10. Exercises

1. Change `MISSING_THRESHOLD` to 0.10 and 0.50. Which columns are kept? How does test AUC change?
2. In `preprocess`, replace `SimpleImputer` with `KNNImputer`. Does it help?
3. Remove the `RobustScaler` step from `preprocess`. What happens to the logistic regression? And to the random forest?
4. Add an SVM (`SVC(probability=True)`) and tune `C` and `gamma` with `RandomizedSearchCV` using `loguniform` distributions (`from scipy.stats import loguniform`).
5. Choose the threshold that maximises Youden's J (sensitivity + specificity − 1) instead of fixing 90% sensitivity.
6. **Leakage experiment:** Do all the preprocessing steps on the *whole* dataset and compute the feature ranking on all the data before tuning. Is the test AUC higher? Why is that result untrustworthy?